In [1]:
import numpy as np
from data.mnist.data_utils import *

In [ ]:
# ----- Load MNIST data -----

In [2]:
X_train, Y_train, X_test, Y_test = load_data()

idxs = np.arange(len(X_train))
np.random.shuffle(idxs)
split_idx = int(np.ceil(len(idxs)*0.8))
X_valid, Y_valid = X_train[idxs[split_idx:]], Y_train[idxs[split_idx:]]
X_train, Y_train = X_train[idxs[:split_idx]], Y_train[idxs[:split_idx]]
print()
print('Set validation data aside')
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', Y_train.shape)
print('Validation data shape: ', X_valid.shape)
print('Validation labels shape: ', Y_valid.shape)

MNIST data loaded:
Training data shape: (60000, 784)
Training labels shape: (60000, 10)
Test data shape: (10000, 784)
Test labels shape: (10000, 10)

Set validation data aside
Training data shape:  (48000, 784)
Training labels shape:  (48000, 10)
Validation data shape:  (12000, 784)
Validation labels shape:  (12000, 10)


In [ ]:
# ----- utils -----

In [32]:
def sigmoid(x, derivative=False):
    if derivative:
        return (np.exp(-x))/((np.exp(-x)+1)**2)
    return 1/(1 + np.exp(-x))


def softmax(x, derivative=False):
    # logit = np.exp(x)이 아닌 이유
    # : 만약 지나치게 큰 값이 원소로 들어가게 된다면, 값이 너무 커서 연산이 되지 않는 오버플로 문제를 일으킬 위험이 있음
    # softmax에 들어가는 원소들에 대하여, 그 원소들의 최댓값을 빼는 것으로 해결 가능!
    exps = np.exp(x - x.max())
    if derivative:
        return exps / np.sum(exps, axis=0) * (1 - exps / np.sum(exps, axis=0))
    return exps / np.sum(exps, axis=0)


def load_batch(X, Y, batch_size, shuffle=True):

    if shuffle:
        permutation = np.random.permutation(X.shape[0])
        X = X[permutation, :]
        Y = Y[permutation, :]
    
    num_steps = int(X.shape[0])//batch_size
    step = 0
    
    while step<num_steps:
        X_batch = X[batch_size*step:batch_size*(step+1)]
        Y_batch = Y[batch_size*step:batch_size*(step+1)]
        step+=1
        yield X_batch, Y_batch

In [33]:
# ----- 2-Layer Neural Network -----

X: (num, input_dim)

W: (input_dim, num_hiddens)

XW: (num, num_hiddens)

In [47]:
class TwoLayerNN:

    def __init__(self, input_dim, num_hiddens, num_classes):
        
        self.input_dim = input_dim
        self.num_hiddens = num_hiddens
        self.num_classes = num_classes
        self.params = self.initialize_parameters(input_dim, num_hiddens, num_classes)

    def initialize_parameters(self, input_dim, num_hiddens, num_classes):
        """
        initializes parameters with Xavier Initialization.
        """
        params = {}
        params["W1"] = np.random.randn(input_dim, num_hiddens)/ np.sqrt(input_dim) 
        params["b1"] = np.zeros((1, num_hiddens))
        params["W2"] = np.random.randn(num_hiddens, num_classes)/ np.sqrt(num_hiddens) 
        params["b2"] = np.zeros((1, num_classes))

        return params
    
    def forward(self, X):
        """
          y = softmax(sigmoid(X W1 + b1) W2 + b2)

        Inputs
        - X: the input matrix of shape (N, D)

        Returns
        - y: the output of the model (N, C)
        - ff_dict: a dictionary with all the fully connected units and activations.
        """
        params = self.params
        
        ff_dict = {}
        ff_dict['z1'] = np.dot(X, params["W1"]) + params["b1"]
        ff_dict['h1'] = sigmoid(ff_dict['z1'])
        ff_dict['z2'] = np.dot(ff_dict['h1'], params["W2"]) + params["b2"]
        ff_dict['h2'] = softmax(ff_dict['z2'])
        y = ff_dict['h2']
        
        return y, ff_dict

    
    def backward(self, X, Y, ff_dict):
        """
        Inputs:
         - X: the input matrix of shape (B, D), where B is the number of examples
              in a mini-batch, D is the feature dimensionality.
         - Y: the matrix of one-hot encoded ground truth classes of shape (B, C),
              where B is the number of examples in a mini-batch, C is the number
              of classes.
         - ff_dict: the dictionary containing all the fully connected units and
              activations.

        Returns:
         - grads: a dictionary containing the gradients of corresponding weights and biases.
      
        y = softmax(sigmoid(X*W1 + b1) W2 + b2)
        
        z1 = W1*X + b1
        h1 = sigmoid(z1)
        z2 = W2*h1 + b2
        h2 = softmax(W2*h1 + b2) = softmax(z2)
        y = h2
        
        L = (y-true_y)**2
        dLdy = 2(y-true_y)
        dLdW2 = dLdy * dydW2 = dLdy * dydz2 * dz2dW2 = 2(y-true_y) * so'(z2)' * h1
        dLdb2 = dLdy * dydb2 = dLdy * dydz2 * dz2db2 = 2(y-true_y) * so'(z2)'
        dLdW1 = dLdy * dydW1 = dLdy * dydz2 * dz2dh1 * dh1dz1 * dz1dW1 = 2(y-true_y) * so'(z2)' * W2 * si'(z1)' * X
        dLdb1 = dLdy * dydb1 = dLdy * dydz2 * dz2dh1 * dh1dz1 * dz1db1 = 2(y-true_y) * so'(z2)' * W2 * si'(z1)'
        """
        z1 = ff_dict['z1'] # (48000, 64)
        h1 = ff_dict['h1'] # (48000, 64)
        z2 = ff_dict['z2'] # (48000, 10)
        h2 = ff_dict['h2'] # (48000, 10)
        dLdy = 2*(h2-Y) # (48000, 10)
        dydz2 = softmax(z2, derivative=True) # (48000, 10)
        dz2dh1 = self.params['W2'] # (64, 10)
        dh1dz1 = sigmoid(z1, derivative=True) # (48000, 64)
        dz1dW1 = X # (48000, 784)    
        dz2dW2 = h1 # (48000, 64)
        
        grads = {}
        grads["dW2"] = dLdy * dydz2 * dz2dW2
        grads["db2"] = dLdy * dydz2
        grads["dW1"] = dLdy * dydz2 * dz2dh1 * dh1dz1 * dz1dW1
        grads["db1"] = dLdy * dydz2 * dz2dh1 * dh1dz1

        
        return grads
    


In [48]:
# ----- Training & Evaluation -----

In [49]:
# model instantiation
model = TwoLayerNN(input_dim=784, num_hiddens=64, num_classes=10)

In [52]:
# 확인용
params = model.initialize_parameters(input_dim=784, num_hiddens=64, num_classes=10)
print('initialize_parameters')
print('W1', params['W1'].shape)
print("b1", params["b1"].shape)
print("W2", params["W2"].shape)
print("b2", params["b2"].shape)

print(' ')

y, ff_dict = model.forward(X_train)
print('forward')
print('z1', ff_dict['z1'].shape)
print('h1', ff_dict['h1'].shape)
print('z2', ff_dict['z2'].shape)
print('h2', ff_dict['h2'].shape)

grads = model.backward(X_train, Y_train, ff_dict)
grads

initialize_parameters
W1 (784, 64)
b1 (1, 64)
W2 (64, 10)
b2 (1, 10)
 
forward
z1 (48000, 64)
h1 (48000, 64)
z2 (48000, 10)
h2 (48000, 10)


ValueError: operands could not be broadcast together with shapes (48000,10) (64,10) 

In [58]:
sigmoid(ff_dict['z1'], derivative=True).shape

(48000, 64)

In [ ]:
# train the model
lr, n_epochs, batch_size = 2.0, 20, 256
model.train(X_train, Y_train, X_valid, Y_valid, lr, n_epochs, batch_size)

In [ ]:
class DeepNeuralNetwork():
    def __init__(self, sizes, epochs=10, l_rate=0.001):
        self.sizes = sizes
        self.epochs = epochs
        self.l_rate = l_rate

        # we save all parameters in the neural network in this dictionary
        self.params = self.initialization()


    def initialization(self):
        # number of nodes in each layer
        input_layer=self.sizes[0]
        hidden_1=self.sizes[1]
        hidden_2=self.sizes[2]
        output_layer=self.sizes[3]

        params = {
            'W1':np.random.randn(hidden_1, input_layer) * np.sqrt(1. / hidden_1),
            'W2':np.random.randn(hidden_2, hidden_1) * np.sqrt(1. / hidden_2),
            'W3':np.random.randn(output_layer, hidden_2) * np.sqrt(1. / output_layer)
        }

        return params

    def forward_pass(self, x_train):
        params = self.params

        # input layer activations becomes sample
        params['A0'] = x_train

        # input layer to hidden layer 1
        params['Z1'] = np.dot(params["W1"], params['A0'])
        params['A1'] = self.sigmoid(params['Z1'])

        # hidden layer 1 to hidden layer 2
        params['Z2'] = np.dot(params["W2"], params['A1'])
        params['A2'] = self.sigmoid(params['Z2'])

        # hidden layer 2 to output layer
        params['Z3'] = np.dot(params["W3"], params['A2'])
        params['A3'] = self.softmax(params['Z3'])

        return params['A3']

    def backward_pass(self, y_train, output):
        '''
            This is the backpropagation algorithm, for calculating the updates
            of the neural network's parameters.

            Note: There is a stability issue that causes warnings. This is 
                  caused  by the dot and multiply operations on the huge arrays.
                  
                  RuntimeWarning: invalid value encountered in true_divide
                  RuntimeWarning: overflow encountered in exp
                  RuntimeWarning: overflow encountered in square
        '''
        params = self.params
        change_w = {}

        # Calculate W3 update
        error = 2 * (output - y_train) / output.shape[0] * self.softmax(params['Z3'], derivative=True)
        change_w['W3'] = np.outer(error, params['A2'])

        # Calculate W2 update
        error = np.dot(params['W3'].T, error) * self.sigmoid(params['Z2'], derivative=True)
        change_w['W2'] = np.outer(error, params['A1'])

        # Calculate W1 update
        error = np.dot(params['W2'].T, error) * self.sigmoid(params['Z1'], derivative=True)
        change_w['W1'] = np.outer(error, params['A0'])

        return change_w

    def update_network_parameters(self, changes_to_w):
        '''
            Update network parameters according to update rule from
            Stochastic Gradient Descent.

            θ = θ - η * ∇J(x, y), 
                theta θ:            a network parameter (e.g. a weight w)
                eta η:              the learning rate
                gradient ∇J(x, y):  the gradient of the objective function,
                                    i.e. the change for a specific theta θ
        '''
        
        for key, value in changes_to_w.items():
            self.params[key] -= self.l_rate * value

    def compute_accuracy(self, x_val, y_val):
        '''
            This function does a forward pass of x, then checks if the indices
            of the maximum value in the output equals the indices in the label
            y. Then it sums over each prediction and calculates the accuracy.
        '''
        predictions = []

        for x, y in zip(x_val, y_val):
            output = self.forward_pass(x)
            pred = np.argmax(output)
            predictions.append(pred == np.argmax(y))
        
        return np.mean(predictions)

    def train(self, x_train, y_train, x_val, y_val):
        start_time = time.time()
        for iteration in range(self.epochs):
            for x,y in zip(x_train, y_train):
                output = self.forward_pass(x)
                changes_to_w = self.backward_pass(y, output)
                self.update_network_parameters(changes_to_w)
            
            accuracy = self.compute_accuracy(x_val, y_val)
            print('Epoch: {0}, Time Spent: {1:.2f}s, Accuracy: {2:.2f}%'.format(
                iteration+1, time.time() - start_time, accuracy * 100
            ))
            
dnn = DeepNeuralNetwork(sizes=[784, 128, 64, 10])
dnn.train(x_train, y_train, x_val, y_val)